# LP–GP Matching: Vineyard Ventures

**Fund:** Vineyard Ventures | **Size:** ~$20M | **Strategy:** FoF — Indian deeptech (AI, hardware, defence, bio) | **Stage:** Pre-seed / Seed | **Vintage:** Fund 1

This notebook loads AI-extracted LP signals from `data/lp_signals.json` and produces a ranked shortlist with actionable outreach recommendations.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 50)

# ── Load data ──────────────────────────────────────────────────────────────
data_path = Path('data/lp_signals.json')
with open(data_path) as f:
    records = json.load(f)

print(f'Loaded {len(records)} LP records')

In [ ]:
# ── Build flat DataFrame ───────────────────────────────────────────────────
rows = []
for r in records:
    s = r['signals']
    sf = r.get('structured_fields', {})
    rows.append({
        'Name':               r['name'],
        'Status':             sf.get('Status', '—'),
        'Location':           sf.get('Location', '—'),
        'LP Type':            s.get('lp_type', '—'),
        'AUM':                s.get('aum_estimate') or '—',
        'Check Size':         s.get('typical_check_size') or sf.get('Check Size', '—'),
        'Fit Score':          s.get('fit_score', 0),
        'India Interest':     s.get('india_interest', 0),
        'FoF Openness':       s.get('fof_openness', 0),
        'Engagement':         s.get('engagement_level', 0),
        'FoF Experience':     s.get('fof_experience', False),
        'Emerging Mgr':       s.get('emerging_manager_preference', False),
        'Deeptech':           s.get('deeptech_interest', False),
        'Sector Agnostic':    s.get('sector_agnostic', False),
        'Preferred Fund Size':s.get('preferred_fund_size') or '—',
        'Fit Rationale':      s.get('fit_rationale', ''),
        'Blockers':           '; '.join(s.get('blockers', [])),
        'Positives':          '; '.join(s.get('positives', [])),
        'Notion URL':         r.get('notion_url', ''),
    })

df = pd.DataFrame(rows).sort_values('Fit Score', ascending=False).reset_index(drop=True)
df.index += 1  # 1-based rank

# Composite score: weighted average of key signals
df['Composite'] = (
    0.40 * df['Fit Score'] +
    0.20 * df['India Interest'] +
    0.20 * df['FoF Openness'] +
    0.20 * df['Engagement']
).round(1)

print('DataFrame built')
df[['Name', 'Status', 'LP Type', 'Fit Score', 'India Interest', 'FoF Openness', 'Engagement', 'Composite']].head(13)

## 1. Ranked Shortlist

In [ ]:
# Tier assignment
def tier(score):
    if score >= 7.5: return 'Tier 1 — Priority'
    if score >= 6.0: return 'Tier 2 — Strong'
    if score >= 4.5: return 'Tier 3 — Moderate'
    return 'Tier 4 — Low'

df['Tier'] = df['Composite'].apply(tier)

display_cols = ['Name', 'Tier', 'Status', 'LP Type', 'AUM', 'Check Size',
                'Fit Score', 'India Interest', 'FoF Openness', 'Engagement', 'Composite']

# Colour-code by tier
tier_colors = {
    'Tier 1 — Priority': 'background-color: #d4edda',
    'Tier 2 — Strong':   'background-color: #d1ecf1',
    'Tier 3 — Moderate': 'background-color: #fff3cd',
    'Tier 4 — Low':      'background-color: #f8d7da',
}

def color_tier(row):
    color = tier_colors.get(row['Tier'], '')
    return [color] * len(row)

df[display_cols].style.apply(color_tier, axis=1)

## 2. Signal Visualisations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: Composite score bar chart ───────────────────────────────────────
ax = axes[0]
names = df['Name'].tolist()
composites = df['Composite'].tolist()
tier_list = df['Tier'].tolist()

color_map = {
    'Tier 1 — Priority': '#2ecc71',
    'Tier 2 — Strong':   '#3498db',
    'Tier 3 — Moderate': '#f39c12',
    'Tier 4 — Low':      '#e74c3c',
}
bar_colors = [color_map[t] for t in tier_list]

bars = ax.barh(names[::-1], composites[::-1], color=bar_colors[::-1], edgecolor='white', height=0.7)
ax.set_xlabel('Composite Score (0–10)', fontsize=11)
ax.set_title('LP Composite Fit Score\n(40% fit · 20% India · 20% FoF · 20% engagement)', fontsize=11)
ax.set_xlim(0, 10)
ax.axvline(7.5, color='#2ecc71', linestyle='--', alpha=0.6, linewidth=1.2, label='Tier 1 threshold')
ax.axvline(6.0, color='#3498db', linestyle='--', alpha=0.6, linewidth=1.2, label='Tier 2 threshold')
ax.axvline(4.5, color='#f39c12', linestyle='--', alpha=0.6, linewidth=1.2, label='Tier 3 threshold')
for bar, score in zip(bars[::-1], composites[::-1]):
    ax.text(score + 0.1, bar.get_y() + bar.get_height() / 2, f'{score}',
            va='center', fontsize=9)
patches = [mpatches.Patch(color=v, label=k) for k, v in color_map.items()]
ax.legend(handles=patches, loc='lower right', fontsize=8)

# ── Right: Heatmap of key signal dimensions ────────────────────────────────
ax2 = axes[1]
dims = ['India\nInterest', 'FoF\nOpenness', 'Engagement', 'Fit\nScore']
data_cols = ['India Interest', 'FoF Openness', 'Engagement', 'Fit Score']
heat_data = df[data_cols].values.astype(float)

im = ax2.imshow(heat_data, aspect='auto', cmap='RdYlGn', vmin=0, vmax=10)
ax2.set_xticks(range(len(dims)))
ax2.set_xticklabels(dims, fontsize=10)
ax2.set_yticks(range(len(names)))
ax2.set_yticklabels(names, fontsize=9)
ax2.set_title('Signal Heatmap by Dimension', fontsize=11)
plt.colorbar(im, ax=ax2, label='Score (0–10)')

for i in range(len(names)):
    for j in range(len(dims)):
        val = heat_data[i, j]
        ax2.text(j, i, f'{int(val)}', ha='center', va='center',
                 fontsize=9, color='black' if 3 < val < 8 else 'white')

plt.tight_layout()
plt.savefig('data/lp_signal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to data/lp_signal_analysis.png')

## 3. Dimension Breakdown per LP

In [ ]:
# Spider/radar chart for top-6 LPs
top6 = df.head(6)
categories = ['Fit Score', 'India Interest', 'FoF Openness', 'Engagement']
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the loop

fig, axes = plt.subplots(2, 3, figsize=(14, 9), subplot_kw=dict(polar=True))
axes = axes.flatten()

colors = ['#2ecc71', '#2ecc71', '#3498db', '#3498db', '#f39c12', '#f39c12']

for idx, (_, row) in enumerate(top6.iterrows()):
    values = [row[c] for c in categories]
    values += values[:1]
    ax = axes[idx]
    ax.plot(angles, values, color=colors[idx], linewidth=2)
    ax.fill(angles, values, color=colors[idx], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=8)
    ax.set_ylim(0, 10)
    ax.set_yticks([2, 4, 6, 8, 10])
    ax.set_yticklabels(['2','4','6','8','10'], fontsize=6, color='grey')
    ax.set_title(f"{row['Name']}\n(composite: {row['Composite']})",
                 fontsize=10, pad=12)

plt.suptitle('Top-6 LP Profiles — Signal Radar', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('data/lp_radar_top6.png', dpi=150, bbox_inches='tight')
plt.show()
print('Radar chart saved to data/lp_radar_top6.png')

## 4. Tier 1 & 2 LP Profiles — Deep Dive

In [ ]:
priority_lps = df[df['Tier'].isin(['Tier 1 — Priority', 'Tier 2 — Strong'])]

for rank, row in priority_lps.iterrows():
    tier_badge = {'Tier 1 — Priority': '🟢', 'Tier 2 — Strong': '🔵'}.get(row['Tier'], '')
    print(f"{'='*72}")
    print(f"#{rank}  {tier_badge}  {row['Name']}  ({row['LP Type']})  |  Composite: {row['Composite']}/10")
    print(f"   Status: {row['Status']}  |  Location: {row['Location']}  |  AUM: {row['AUM']}  |  Check: {row['Check Size']}")
    print(f"   Fit: {row['Fit Score']}/10  |  India: {row['India Interest']}/10  |  FoF: {row['FoF Openness']}/10  |  Engagement: {row['Engagement']}/10")
    print(f"   FoF Exp: {row['FoF Experience']}  |  Emerging-mgr preference: {row['Emerging Mgr']}  |  Deeptech: {row['Deeptech']}  |  Sector-agnostic: {row['Sector Agnostic']}")
    print(f"   Pref fund size: {row['Preferred Fund Size']}")
    print(f"\n   RATIONALE: {row['Fit Rationale']}")
    print(f"\n   POSITIVES: {row['Positives']}")
    if row['Blockers']:
        print(f"   BLOCKERS:  {row['Blockers']}")
    print(f"\n   Notion: {row['Notion URL']}")
    print()

print('='*72)

## 5. Outreach Recommendations

In [ ]:
recommendations = [
    {
        'Rank': 1,
        'LP': 'Dziugas',
        'Tier': 'Tier 1',
        'Urgency': 'High — anchor candidate',
        'Next Action': 'Send working doc outlining LP involvement model + GP co-access rights. Follow up end of Jan when exit process closes.',
        'Key Message': 'First-believer framing — small intimate fund where he calls me, not a faceless institution. Anchor $5–10M post-exit.',
    },
    {
        'Rank': 2,
        'LP': 'GEM',
        'Tier': 'Tier 1',
        'Urgency': 'High — in diligence, warm to India',
        'Next Action': 'Provide India benchmarking: intro to Prime, Sauce, Northpoint. Follow up with differentiated emerging GP case study.',
        'Key Message': 'Position Vineyard as the missing India mid-tier access layer. Emphasise consistency, brand-building, founder references.',
    },
    {
        'Rank': 3,
        'LP': 'Alber blanc',
        'Tier': 'Tier 1',
        'Urgency': 'Medium — Qualified, responsive',
        'Next Action': 'Arrange intro call / meeting in NYC. Share deck and emerging manager track record comp.',
        'Key Message': 'Validate India thesis with simple narrative. Lean on his openness to first-time GPs and FoF experience.',
    },
    {
        'Rank': 4,
        'LP': 'Weizmann Institute endowment',
        'Tier': 'Tier 2',
        'Urgency': 'Medium — In Nurture, deeptech + India exposure',
        'Next Action': 'Arrange LP meeting with detailed India deeptech landscape. Address valuation and DPI concerns with exit data from portfolio GPs.',
        'Key Message': 'Fund 1 specialist — ideal entry point. Differentiated mandate vs Z47/Accel. Concrete DPI narrative from GP portfolio.',
    },
    {
        'Rank': 5,
        'LP': 'Rezayat',
        'Tier': 'Tier 2',
        'Urgency': 'Low-Medium — Nurture, FoF + EM experience',
        'Next Action': 'Leverage James Eggington relationship. Build India interest with deeptech angle given industrial background.',
        'Key Message': 'Industrial / defence deeptech alignment. Relationship-driven, small team — play up intimacy of fund model.',
    },
]

rec_df = pd.DataFrame(recommendations).set_index('Rank')
rec_df.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'})

## 6. LPs to Deprioritise

In [ ]:
low_tier = df[df['Tier'] == 'Tier 4 — Low'][['Name', 'LP Type', 'Composite', 'Fit Rationale']]
print('Tier 4 LPs — Low priority / strategic misalignment:')
display(low_tier)

print('\nTier 3 LPs — Moderate fit, nurture only:')
moderate = df[df['Tier'] == 'Tier 3 — Moderate'][['Name', 'LP Type', 'Composite', 'Blockers']]
display(moderate)

## 7. Summary Statistics

In [ ]:
print('=== VINEYARD VENTURES — LP SHORTLIST SUMMARY ===')
print(f'Total LPs analysed: {len(df)}')
print()
tier_counts = df['Tier'].value_counts().sort_index()
for tier, count in tier_counts.items():
    print(f'  {tier}: {count} LP(s)')
print()
print(f'Average composite score: {df["Composite"].mean():.1f}/10')
print(f'Average India interest:  {df["India Interest"].mean():.1f}/10')
print(f'Average FoF openness:    {df["FoF Openness"].mean():.1f}/10')
print(f'Average engagement:      {df["Engagement"].mean():.1f}/10')
print()
print(f'LPs with FoF experience:          {df["FoF Experience"].sum()}/{len(df)}')
print(f'LPs preferring emerging managers: {df["Emerging Mgr"].sum()}/{len(df)}')
print(f'LPs with deeptech interest:       {df["Deeptech"].sum()}/{len(df)}')
print()
print('Status breakdown:')
status_breakdown = df.groupby('Status')['Name'].apply(list)
for status, names in status_breakdown.items():
    print(f'  {status}: {", ".join(names)}')